In [1]:
%matplotlib widget

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import tqdm

import mne
from mne.io import concatenate_raws, read_raw_edf
import numpy as np

from bscarlos.settings import RAW_KISPI_DATA_FOLDER
from bscarlos.settings import PROCESSED_KISPI_DATA_FOLDER

## EDA 
Here we explore the EEG data from KISPI patients.  We first get a general sense of the data, length of each recording and number of (single & bipolar) channels.  The next steps will be to analyze the data with more statsitical methods, for this in particular we explore `tsfresh` package which is used to perform feature engineering on time-series and other sequential data<sup>1</sup> similar to ours.


--- 


[1] Christ, M., Braun, N., Neuffer, J. and Kempa-Liehr A.W. (2018). Time Series FeatuRe Extraction on basis of Scalable Hypothesis tests (tsfresh – A Python package). Neurocomputing 307 (2018) 72-77, doi: 10.1016/j.neucom.2018.03.067.

### Convert .edf to parquet files
In order to better handle the dataset it would be advantagous to create a dataframe using trusty `pandas`, however we first needed to convert our EEG files from .edf format into parquet files.  To accomplish this we used the `edf2parquet` a simple utility package to convert EDF/EDF+ files into Apache Parquet format while preserving the EDF file header information and signal headers metadata information then we will be able to read the files directly<sup>2</sup>. In addition we create a data_attributes.csv file with all the important attributes we unvail.

---


[2] [edf2parquet repository](https://github.com/NarayanSchuetz/edf2parquet/tree/main)

In [3]:
import pytz
from edf2parquet.converters import AdvancedEdfToParquetConverter


my_edf_file = "C:\\Users\\c_arz\\Documents\\KISPI\\Burst_Suppression_Project\\Thesis\\UnsuperDL-EEG-BSUPP\\data\\raw_kispi\\SE008_120418U-A.edf"  # REPLACE WITH YOUR EDF FILE PATH
my_parquet_output_dir = "C:\\Users\\c_arz\\Documents\\KISPI\\Burst_Suppression_Project\\Thesis\\UnsuperDL-EEG-BSUPP\\data\\raw_kispi"  # REPLACE WITH YOUR PARQUET OUTPUT DIRECTORY

converter = AdvancedEdfToParquetConverter(edf_file_path=my_edf_file,  # path to the EDF file
                                          exclude_signals=["Audio"],  # list of signals to exclude from the conversion
                                          parquet_output_dir=my_parquet_output_dir,  # path to the output directory (will be created if not exists)
                                          group_by_sampling_freq=True,  # whether to group signals with same sampling frequency into single parquet files
                                          datetime_index=True,  # whether to automatically add a pd.DatetimeIndex to the resulting parquet files
                                          local_timezone=(pytz.timezone("Europe/Zurich"), pytz.timezone("Europe/Zurich")),  # specifies the timezone of the EDF file and the timezone of the start_date in the EDF file (should be the same for most cases)
                                          compression_codec="GZIP" # compression codec to use for the resulting parquet files
                                          )

converter.convert()

c:\Users\c_arz\Documents\KISPI\Burst_Suppression_Project\Thesis\UnsuperDL-EEG-BSUPP\.venv\Lib\site-packages\edf2parquet\readers.py:236: FutureWarning: 'N' is deprecated and will be removed in a future version, please use 'ns' instead.
  time_points = pd.date_range(


In [18]:
from edf2parquet.converters import AdvancedEdfToParquetConverter
import os
from tqdm import tqdm

# Define the input and output folders
input_folder = RAW_KISPI_DATA_FOLDER
output_folder = PROCESSED_KISPI_DATA_FOLDER

# Create the output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Get a list of all .edf files in the input folder
filenames = [filename for filename in os.listdir(input_folder) if filename.endswith('.edf')]

# Iterate and convert all .edf files
for filename in tqdm(filenames, desc='Converting EDF to Parquet'):
    # Split the filename at '_' and use the first part as the output filename
    output_filename = filename.split('_')[0] + '.parquet'

    # Create the full output file path
    output_file_path = os.path.join(output_folder, output_filename)

    # Create an instance of the converter
    converter = AdvancedEdfToParquetConverter(
        edf_file_path=os.path.join(input_folder, filename),
        parquet_output_dir=output_folder,
        exclude_signals=["EEG A1", "EEG A2", "EKG", "EOG", "EMG", "PHO"],
        group_by_sampling_freq=True,
        datetime_index=True,
        local_timezone=(pytz.timezone("Europe/Zurich"), pytz.timezone("Europe/Zurich")),
        compression_codec="GZIP"
    )
    
    # Convert the EDF file to Parquet and save it to the output folder
    converter.convert()

Converting EDF to Parquet:   0%|          | 0/4 [00:00<?, ?it/s]c:\Users\c_arz\Documents\KISPI\Burst_Suppression_Project\Thesis\UnsuperDL-EEG-BSUPP\.venv\Lib\site-packages\edf2parquet\readers.py:236: FutureWarning: 'N' is deprecated and will be removed in a future version, please use 'ns' instead.
  time_points = pd.date_range(
Converting EDF to Parquet:  25%|██▌       | 1/4 [00:47<02:23, 47.92s/it]c:\Users\c_arz\Documents\KISPI\Burst_Suppression_Project\Thesis\UnsuperDL-EEG-BSUPP\.venv\Lib\site-packages\edf2parquet\readers.py:236: FutureWarning: 'N' is deprecated and will be removed in a future version, please use 'ns' instead.
  time_points = pd.date_range(
Converting EDF to Parquet:  50%|█████     | 2/4 [00:49<00:40, 20.48s/it]c:\Users\c_arz\Documents\KISPI\Burst_Suppression_Project\Thesis\UnsuperDL-EEG-BSUPP\.venv\Lib\site-packages\edf2parquet\readers.py:236: FutureWarning: 'N' is deprecated and will be removed in a future version, please use 'ns' instead.
  time_points = pd.date_r

#### Read a converted Parquet file using handy pandas.

In [20]:
# abolute path to the parquet file
my_parquet_file_path = "C:\\Users\\c_arz\\Documents\\KISPI\\Burst_Suppression_Project\\Thesis\\UnsuperDL-EEG-BSUPP\\data\\raw_kispi\\56391_2012-04-17_256.0.parquet"

# create dataframe from parquet file
df = pd.read_parquet(my_parquet_file_path)

#### Read a converted Pareut file using the ParquetReader class directly

In [21]:
from edf2parquet.readers import ParquetReader

reader = ParquetReader(parquet_file_path=my_parquet_file_path)
df = reader.get_pandas_dataframe(set_timezone=True) # (Note that here we set a timezone which is different to when we used plain pandas)

In [22]:
reader.get_file_header()

{'technician': '',
 'recording_additional': '',
 'patientname': 'X',
 'patient_additional': '',
 'patientcode': '56391',
 'equipment': 'Deltamed',
 'admincode': 'SE008 120418U-A',
 'sex': '',
 'startdate': Timestamp('2012-04-17 17:02:08+0200', tz='Europe/Zurich'),
 'birthdate': '',
 'gender': '',
 'tz_recording': 'Europe/Zurich',
 'tz_startdatetime': 'Europe/Zurich'}

In [14]:
reader.get_signal_headers()

[{'label': 'EEG Fp1',
  'dimension': 'uV',
  'sample_rate': 256.0,
  'sample_frequency': 256.0,
  'physical_max': 5865.0,
  'physical_min': -5865.0,
  'digital_max': 32767,
  'digital_min': -32768,
  'prefilter': 'HP:10Hz LP:100Hz',
  'transducer': 'AgAgCl electrode'},
 {'label': 'EEG Fp2',
  'dimension': 'uV',
  'sample_rate': 256.0,
  'sample_frequency': 256.0,
  'physical_max': 5865.0,
  'physical_min': -5865.0,
  'digital_max': 32767,
  'digital_min': -32768,
  'prefilter': 'HP:10Hz LP:100Hz',
  'transducer': 'AgAgCl electrode'},
 {'label': 'EEG F7',
  'dimension': 'uV',
  'sample_rate': 256.0,
  'sample_frequency': 256.0,
  'physical_max': 5865.0,
  'physical_min': -5865.0,
  'digital_max': 32767,
  'digital_min': -32768,
  'prefilter': 'HP:10Hz LP:100Hz',
  'transducer': 'AgAgCl electrode'},
 {'label': 'EEG F3',
  'dimension': 'uV',
  'sample_rate': 256.0,
  'sample_frequency': 256.0,
  'physical_max': 5865.0,
  'physical_min': -5865.0,
  'digital_max': 32767,
  'digital_min': -3

In [31]:
new_parquet_file_path ="C:\\Users\\c_arz\\Documents\\KISPI\\Burst_Suppression_Project\\Thesis\\UnsuperDL-EEG-BSUPP\\data\\processed_kispi\\56391_2012-04-17_256.0.parquet"

In [32]:
df = pd.read_parquet(new_parquet_file_path)

# Print the first 5 rows
print(df.head())

# Print column names and their data types
print(df.info())

# Get summary statistics of numerical columns
print(df.describe())

                               EEG Fp1    EEG Fp2      EEG F7      EEG F3  \
2012-04-17 17:02:08.000000  107.303505  78.486382  163.326843  136.478592   
2012-04-17 17:02:08.003906  107.303505  78.665367  164.400772  136.120621   
2012-04-17 17:02:08.007812  108.914398  80.634239  167.980545  138.626465   
2012-04-17 17:02:08.011718  108.914398  80.992218  167.443573  138.805450   
2012-04-17 17:02:08.015625  108.556419  81.529182  166.727631  137.552536   

                              EEG Fz      EEG F4      EEG F8      EEG T3  \
2012-04-17 17:02:08.000000 -6.354085  123.591438  143.638138  102.649803   
2012-04-17 17:02:08.003906 -6.354085  121.622566  142.922180  101.217896   
2012-04-17 17:02:08.007812 -5.996109  123.412453  145.428009  103.723732   
2012-04-17 17:02:08.011718 -5.459144  123.591438  146.143967  102.470818   
2012-04-17 17:02:08.015625 -3.311284  124.486382  145.964981  101.575874   

                               EEG C3     EEG Cz      EEG C4    EEG T4  \
2012-0

#### TO DO: 
with the following code make a data_attributes.csv file 

In [ ]:
import pandas as pd
import pyarrow.parquet as pq

def create_data_attributes_csv(parquet_file, csv_file):
  """
  Creates a CSV file named 'data_attributes_kispi.csv' from a Parquet file 
  containing patient data.

  Args:
    parquet_file: Path to the Parquet file.
    csv_file: Path to the output CSV file.
  """

  # Read the Parquet file into a Pandas DataFrame
  df = pd.read_parquet(parquet_file)

  # Assuming the Parquet file has a 'patient_id' column
  unique_patient_ids = df['patient_id'].unique()

  data_attributes = []
  for i, patient_id in enumerate(unique_patient_ids):
    patient_df = df[df['patient_id'] == patient_id]
    start_sample = patient_df.index.min()
    end_sample = patient_df.index.max()
    # Assuming the sampling rate is constant for each patient and stored in a column 'sample_rate'
    sample_rate = patient_df['sample_rate'].iloc[0] 
    data_attributes.append([f"Pat{i:02d}", patient_id, start_sample, end_sample, sample_rate])

  # Create a DataFrame from the extracted data
  data_attributes_df = pd.DataFrame(data_attributes, 
                                   columns=["unique_patient_id", "patient_id", 
                                            "start_sample", "end_sample", "sample_rate"])

  # Save the DataFrame to a CSV file
  data_attributes_df.to_csv(csv_file, index=False)

# Example usage:
parquet_file = "path/to/your/parquet_file.parquet"  # Replace with your Parquet file path
csv_file = "data_attributes_kispi.csv"
create_data_attributes_csv(parquet_file, csv_file)

#### Read parquet files and create patient_data dictionary

In [29]:
from bscarlos.data.data_preprocessing import DataPreprocessing

data_preprocess = DataPreprocessing()

data_files = sorted(list(PROCESSED_KISPI_DATA_FOLDER.glob("*.parquet")))

patient_data = data_preprocess.create_patient_data_dict(data_files, data_attributes=None)

patient_data.keys()


  0%|          | 0/4 [00:00<?, ?it/s]


AttributeError: 'NoneType' object has no attribute 'loc'